# Data Cleaning File for the semester project, ECSE-4840

### .FIT reader utility functions

In [ ]:
import pandas as pd
from fitparse import FitFile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

directory = Path("gpx")

def split_ride(df, trim_s=10, max_gap_s=2):
    df = df.sort_values("timestamp").reset_index(drop=True).copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    stationary = ((df["power"] == 0) & (df["enhanced_speed"] == 0))
    dt = df["timestamp"].diff().dt.total_seconds()
    gap = dt.gt(max_gap_s)
    df = df.loc[~stationary].copy()

    dt = df["timestamp"].diff().dt.total_seconds()
    segment_id = dt.gt(max_gap_s).cumsum()

    segments = []

    for _, segment in df.groupby(segment_id):
        segment = segment.copy()

        start = segment["timestamp"].iloc[0]
        end = segment["timestamp"].iloc[-1]

        segment = segment.loc[(segment["timestamp"] >= start + pd.Timedelta(seconds=trim_s)) & (segment["timestamp"] <= end - pd.Timedelta(seconds=trim_s))].copy()

        if segment.empty:
            continue

        segment["elapsed_time_s"] = (segment["timestamp"] - segment["timestamp"].iloc[0]).dt.total_seconds()

        duration = (segment["timestamp"].iloc[-1] - segment["timestamp"].iloc[0]).total_seconds()

        if duration < 300:
            continue

        segment["elapsed_time_s"] = (segment["timestamp"] - segment["timestamp"].iloc[0]).dt.total_seconds()

        segment["window_id"] = (segment["elapsed_time_s"] // 300).astype(int)

        for _, window in segment.groupby("window_id"):
            window = window.copy()

            duration = (window["timestamp"].iloc[-1] - window["timestamp"].iloc[0]).total_seconds()

            if duration < 299:
                continue

            window = window.set_index("timestamp")
            window = window.resample("1s").mean(numeric_only=True)
            window = window.interpolate(method="time", limit=2)

            window = window.iloc[:300].reset_index()

            if len(window) != 300:
                continue

            cols = ["cadence", "power", "heart_rate", "enhanced_altitude"]

            if window[cols].isna().any().any():
                continue

            window = window.drop(columns=["window_id"], errors="ignore")
            window["elapsed_time_s"] = np.arange(300)

            segments.append(window)

    return segments

def read_fit(filepath):
    fitfile = FitFile(filepath)
    records = []

    for record in fitfile.get_messages("record"):
        row = {}

        for field in record:
            row[field.name] = field.value

        records.append(row)

    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    segments = split_ride(df)
    len_segments = len(segments)

    for i in range(len_segments):
        segments[i]["elapsed_time_s"] = (segments[i]["timestamp"] - segments[i]["timestamp"].iloc[0]).dt.total_seconds()
        segments[i] = segments[i][["cadence", "power", "heart_rate", "elapsed_time_s", "enhanced_altitude"]]

    return segments

def list_file_names(directory, extension=None, recursive=False):
    
    directory = Path(directory).expanduser()

    files = directory.rglob('*') if recursive else directory.iterdir()
    extension = f".{extension.lstrip('.').lower()}" if extension else None
    return sorted(
        path.name for path in files
        if path.is_file() and (extension is None or path.suffix.lower() == extension)
    )



### Read .FIT and Organize into Dataframes, then storing as H5

In [7]:
filenames = list_file_names(directory, extension="fit", recursive=False)
rides = {}
segment_count = 0

for i in range(len(filenames)):
    filepath = "{}/{}".format(directory, filenames[i])
    segments = read_fit(filepath)
    rides[f"{i:03d}"] = segments
    segment_count += len(segments)

print(f"Total segments: {segment_count}")

Total segments: 404


### H5 data saver

In [9]:
import h5py

def save_rides_h5(rides, filepath):
    filepath = Path(filepath)
    with h5py.File(filepath, "w") as h5:
        h5.attrs["format"] = "ride_segments_v1"
        for ride_id, segments in rides.items():
            ride_group = h5.create_group(str(ride_id))
            for i, df in enumerate(segments):
                if not isinstance(df, pd.DataFrame):
                    raise TypeError(f"Ride {ride_id}, segment {i} is not a DataFrame")
                group = ride_group.create_group(f"{i:03d}")
                group.attrs["columns"] = np.asarray([str(c) for c in df.columns], dtype=h5py.string_dtype("utf-8"))
                for j, col in enumerate(df.columns):
                    series = df[col]
                    column_group = group.create_group(f"col_{j:03d}")
                    column_group.attrs["name"] = str(col)
                    if pd.api.types.is_datetime64_any_dtype(series):
                        column_group.attrs["kind"] = "datetime"
                        column_group.create_dataset("values", data=series.astype("datetime64[ns]").array.asi8)
                    elif pd.api.types.is_timedelta64_dtype(series):
                        column_group.attrs["kind"] = "timedelta"
                        column_group.create_dataset("values", data=series.astype("timedelta64[ns]").array.asi8)
                    elif pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
                        column_group.attrs["kind"] = "numeric"
                        column_group.attrs["dtype"] = str(series.dtype)
                        values = series.to_numpy()
                        if values.dtype.kind == "O":
                            values = series.to_numpy(dtype="float64", na_value=np.nan)
                            column_group.attrs["dtype"] = "float64"
                        column_group.create_dataset("values", data=values, compression="gzip")
                    else:
                        nonmissing = series.dropna()
                        if not nonmissing.map(lambda x: isinstance(x, str)).all():
                            raise TypeError(f"Unsupported object values in ride {ride_id}, segment {i}, column {col!r}")
                        column_group.attrs["kind"] = "string"
                        missing = series.isna().to_numpy()
                        values = ["" if pd.isna(v) else v for v in series]
                        column_group.create_dataset("values", data=np.asarray(values, dtype=h5py.string_dtype("utf-8")))
                        column_group.create_dataset("missing", data=missing)

save_rides_h5(rides, "cycling_data.h5")
